# RAG-Powered Course Tutor System

## Project Overview
This repository implements a **Retrieval-Augmented Generation (RAG)** pipeline designed to serve as an intelligent tutor for educational content. The system processes course transcripts, indexes them into a vector database, and leverages a local Large Language Model (LLM) to provide context-accurate answers to student inquiries based strictly on the provided syllabus.

## Tech Stack
- **Orchestration**: [LangChain](https://www.langchain.com/) (LCEL)
- **Vector Database**: [ChromaDB](https://www.trychroma.com/)
- **Embeddings**: HuggingFace `all-MiniLM-L6-v2`
- **Inference**: `microsoft/phi-3-mini-4k-instruct` (via Transformers)
- **Document Processing**: PyPDF & Markdown Header Splitting

## System Architecture

### 1. Ingestion & Pre-processing
- **Structural Splitting**: Uses `MarkdownHeaderTextSplitter` to maintain document hierarchy and metadata.
- **Recursive Chunking**: Implements `TokenTextSplitter` to ensure chunks align with model context window limits (500 tokens with 50-token overlap).

### 2. Retrieval Strategy
- **Vectorization**: Documents are embedded into a high-dimensional space for semantic search.
- **MMR (Maximum Marginal Relevance)**: The retriever is configured with `search_type="mmr"` and `lambda_mult=0.7` to balance relevance with information diversity, reducing redundancy in the retrieved context.

### 3. Generation Chain (LCEL)
- **Query Rephrasing**: A dedicated prompt clarifies student questions before retrieval.
- **Context Injection**: The LLM is constrained via a system prompt to act as a Teaching Assistant, preventing hallucinations by restricting answers to the retrieved snippets.
- **Streaming**: Supports real-time token streaming for a better user experience.

## Installation & Setup

```bash
pip install langchain langchain-community pypdf chromadb transformers sentence-transformers langchain_huggingface
```

## How to Use
1. **Load Data**: Place your course PDF in the workspace.
2. **Initialize Vector Store**: Run the embedding cells to create the `./vector_db` directory.
3. **Query**: Use the `chain_retrieving` object to pass a dictionary containing the student's question metadata.


```

## import Libraries And Files

In [ ]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install pypdf
!pip install -U langchain-text-splitters

In [ ]:
!pip install chromadb

In [ ]:
# from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI
# from google.colab import userdata

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
import copy

In [ ]:
loader_pdf = PyPDFLoader("/content/Introduction_to_Tableau.pdf")
loader_pdf

In [ ]:
pages = loader_pdf.load()
len(pages)

49

In [ ]:
pages_copy = copy.deepcopy(pages)

In [ ]:
text = pages_copy[0].page_content.replace('\n', ' ')
print(text)

# Introduction to Tableau  ## Welcome to Tableau  Hi, everyone.  I'm Ned and I'll be your instructor for this  course.  Tableau is an invaluable tool.  One needs to learn on their journey to become a  successful business intelligence analyst or  data scientist.  The art of these professions is storytelling  using data to tell stories and convince top  management of the right course of action.  By completing this part of the program, you  will know how to create charts and dashboards  in tableaux.  This is an essential step on your way to a data  scientist role.    ## Why use Tableau: Make your data make an impact  Tableau has grown to become one of the most  popular business intelligence tools in the  entire world.  It is A B I software that allows non technical  users to visualize their data and work with it  almost immediately lowering,  know how barriers dramatically in the past.  Business analysts needed the help of it  personnel who could assist them in gathering  raw data and pre

In [ ]:
pages_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=
        [("#", "Course Title"),
        ("##", "Lecture Title")]
)


In [ ]:
string_list_concat = []

string_list_concat =  " ".join(i.page_content for i in pages_copy) # take the page_content attribute of each i

In [ ]:
docs_list_md_split = pages_splitter.split_text(string_list_concat)

In [ ]:
len(docs_list_md_split)

22

In [ ]:
for doc in docs_list_md_split:
    doc.page_content = doc.page_content.replace('\n', ' ')

In [ ]:
docs_list_md_split[0]

Document(metadata={'Course Title': 'Introduction to Tableau', 'Lecture Title': 'Welcome to Tableau'}, page_content="Hi, everyone. I'm Ned and I'll be your instructor for this course. Tableau is an invaluable tool. One needs to learn on their journey to become a successful business intelligence analyst or data scientist. The art of these professions is storytelling using data to tell stories and convince top management of the right course of action. By completing this part of the program, you will know how to create charts and dashboards in tableaux. This is an essential step on your way to a data scientist role.")

In [ ]:
string_list_split = [i.page_content for i in docs_list_md_split]
string_list_split[2]

"Ok, guys, it is time to get started with Tableaux. Let's type Tableau public in Google. As you can see the first result, we have points to Tableau's website at www dot tableau dot com. I'll click on the link and this will direct me to the Tableau public Domain. It shouldn't be too difficult to download Tableau from here if you are wondering why we searched for Tableau public. The reason is quite trivial. This is Tableau's free version if you don't have a paid subscription for Tableaux. This is an excellent alternative. You can practice with most of the program's functionalities and you don't have to pay Tableau's annual fee. So it is up to you. You can either use Tableau public for free or pay for Tableau's desktop version. Both options would allow you to follow along. There are some issues when you want to integrate Tableau public and programming languages like R, Python and SQL to do that, you'll need Tableau desktop. But for now, Tableau public will do just fine and allow us to pra

In [ ]:
inputs = [{"lecture_transcript": s} for s in string_list_split]  # Dict the context for each lecture
inputs[5]

{'lecture_transcript': "All right, excellent. It is time to continue our adventure in Tableaux. In this lesson, we'll create our first visualization and it is going to be awesome ready. Let's get right into it. Then as you can see the workspace area is empty right now, we've already loaded the GDP data file and we can see that here. Actually, let's open the GDP data XL file for a second. I want to make sure you are familiar with its structure here. It is we have a few blank rows but tableau took care of them. Then we have a column with country names, a column indicating that this is GDP data and several columns with GDP figures for each of these countries. And this is the data sheet we are using right now. Perfect. Let's go back to tableaux. The way data is organized here is rather interesting. Our attention should be focused on the dimensions and measures part of the screen. First off, we notice that tableau has been very smart and managed to organize our data. Categorical variables a

the content above is cleaned but to clean it more we put an LLM to look after us

cleaner LLM

Using TinyLlama model

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from langchain_community.llms import HuggingFacePipeline

In [ ]:
from transformers import pipeline


llm = pipeline(
    "text-generation",
   # model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    model="microsoft/phi-3-mini-4k-instruct",
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200,
    do_sample=False,
    repetition_penalty=1.2
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
# from langchain_core.messages import SystemMessage
# from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

# PROMPT_FORMATTING_S = ( '''
#     "you are a words correcter only "
#     "Do not add or invent any sentences, words, or information."
#     "Do not expand any part of the original text."
# '''
# )

# PROMPT_TEMPLATE_FORMATTING_H = """ This is the transcript:
# {lecture_transcript}
# """

# # Create the components
# prompt_formatting_s = SystemMessage(content=PROMPT_FORMATTING_S)
# prompt_template_formatting_h = HumanMessagePromptTemplate.from_template(PROMPT_TEMPLATE_FORMATTING_H)

# chat_prompt_template_formatting = ChatPromptTemplate(
#     messages=[prompt_formatting_s, prompt_template_formatting_h]
# )

In [ ]:
# from langchain_community.llms import HuggingFacePipeline
# from langchain_core.output_parsers import StrOutputParser


# chat = HuggingFacePipeline(pipeline=llm)
# str_output_parser = StrOutputParser()

# # chain_formatting = (
# #     chat_prompt_template_formatting
# #     | chat
# #     | str_output_parser
# # )

/tmp/ipykernel_6266/2164643124.py:5: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  chat = HuggingFacePipeline(pipeline=llm)


In [ ]:
# results = chain_formatting.batch(inputs)

In [ ]:
# inputs[0]

In [ ]:
# print(results[0]) # Shows the cleaned version of the first lecture

In [ ]:
# from langchain_core.documents import Document

# docs_list_md_split = [Document(page_content=txt) for txt in results]

Splitters

In [ ]:
from langchain_text_splitters import TokenTextSplitter

In [ ]:
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

In [ ]:
docs_list_tokens_split = token_splitter.split_documents(docs_list_md_split)

In [ ]:
print(len(docs_list_md_split))
print(len(docs_list_tokens_split))

22
42


In [ ]:
print(docs_list_tokens_split[0].page_content)

Hi, everyone. I'm Ned and I'll be your instructor for this course. Tableau is an invaluable tool. One needs to learn on their journey to become a successful business intelligence analyst or data scientist. The art of these professions is storytelling using data to tell stories and convince top management of the right course of action. By completing this part of the program, you will know how to create charts and dashboards in tableaux. This is an essential step on your way to a data scientist role.


Embeddings

In [ ]:
!pip install sentence-transformers langchain_huggingface

In [ ]:
#!pip install chromadb

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=docs_list_tokens_split,
    embedding=embedding,
    persist_directory="./intro-to-tableau"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "lambda_mult": 0.7}
)

In [ ]:
question="How do I connect Tableau to an Excel file?"

In [ ]:
retrieved_docs = retriever.invoke(question)
for doc in retrieved_docs:
    print(doc.page_content)
    print(f"Lecture Title is {doc.metadata["Lecture Title"]}")
    print("-" * 40)

Right. Great. Here is our freshly installed version of Tableaux. I am sure you are anxious to create some fascinating visualizations. So let's get started. First off, we need to learn how to connect Tableau to the data source we will be working with. There are two options, we can either create a connection to a file or a server. Of course, we'll choose one of the two depending on where the data is. Let's connect Tableau to a Microsoft Excel file in general. Every time we use a source file in one of the lectures, you will be able to find it in the supplemental resources section. Just open your course curriculum and download the available files for that lesson. See, OK, great. I'll select the file called GDP data and under connections. I can now see that Tableau opened the file. Our source has three sheets, data, metadata, countries and metadata indicators. What we usually have to do is choose the worksheet we'll need and drag into the upper part of the screen where drag sheets here is w

Generate

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
PROMPT_CREATING_QUESTION = (
    "Given a student's question, rephrase it to make it as clear and specific as possible for information retrieval."
)

PROMPT_RETRIEVING_S = (
    "You are an expert teaching assistant for a Tableau course. Answer questions accurately and clearly, using only the retrieved course materials. If you are unsure, let the student know and do not invent information."
)


PROMPT_TEMPLATE_RETRIEVING_H = (
    "Here is some relevant course content:\n{retrieved_content}\n\n"
    "Student's question: {question}\n"
    "Using only the content above, provide a clear, complete answer."
)

In [ ]:
prompt_creating_question = PromptTemplate.from_template(template=PROMPT_CREATING_QUESTION)
prompt_retrieving_s =      SystemMessage(content=PROMPT_RETRIEVING_S)
prompt_template_retrieving_h = HumanMessagePromptTemplate.from_template(template=PROMPT_TEMPLATE_RETRIEVING_H)

chat_prompt_template_retrieving = ChatPromptTemplate(        #  Builds the full prompt for your LLM
    messages=[prompt_retrieving_s,
              prompt_template_retrieving_h]

    )
chat_prompt_template_retrieving

ChatPromptTemplate(input_variables=['question', 'retrieved_content'], input_types={}, partial_variables={}, messages=[SystemMessage(content='You are an expert teaching assistant for a Tableau course. Answer questions accurately and clearly, using only the retrieved course materials. If you are unsure, let the student know and do not invent information.', additional_kwargs={}, response_metadata={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question', 'retrieved_content'], input_types={}, partial_variables={}, template="Here is some relevant course content:\n{retrieved_content}\n\nStudent's question: {question}\nUsing only the content above, provide a clear, complete answer."), additional_kwargs={})])

Chain

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser


chat = HuggingFacePipeline(pipeline=llm)
str_output_parser = StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


# You can't send a StringPromptValue to the rest of the chain—you must convert it to a string

In [ ]:
chain = chain | {
    'retrieved_content': retriever,
    'question' : RunnablePassthrough()   # we use passthrought to pass the question (str) unchanged

}
chain_retrieving = (
    prompt_creating_question
    | RunnableLambda(lambda x: x.text)
    | {
        'retrieved_content': retriever,   # This variable name matches the prompt requirement!
        'question': RunnablePassthrough()
      }
    | chat_prompt_template_retrieving
    | chat
    | str_output_parser

)

In [ ]:
result = chain_retrieving.invoke({
    "question_lecture": "Adding a custom calculation",
    "question_title": "Why are we using SUM here? It's unclear to me.",
    "question_body": "This question refers to calculating the GM%."
})
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


System: You are an expert teaching assistant for a Tableau course. Answer questions accurately and clearly, using only the retrieved course materials. If you are unsure, let the student know and do not invent information.
Human: Here is some relevant course content:
[Document(metadata={'Lecture Title': "Exploring Tableau's interface", 'Course Title': 'Introduction to Tableau'}, page_content=" Lets you break a view into a series of pages. So you can better analyze how a specific field affects the rest of the data in a view we'll use the filter shelf when working with filters and filtering our data. The marks shelf on the other hand contains functionalities related to coloring size labels and so on. This lesson was a quick overview of Tableau's interface. I am sure now you have a better idea of what you see in front of you. When you open the program in the lessons to come, we'll continue to explore Tableau's functionalities and you'll learn a ton about each of the buttons we mentioned he

In [ ]:
result_streamed = chain_retrieving.stream({
    "question_lecture": "Adding a custom calculation",
    "question_title": "Why are we using tablea for? It's unclear to me.",
    "question_body": "This question refers to pupose of tableau."
})
for chunk in result_streamed:
    print(chunk, end="")